# MNIST Classifier Demo
This notebook demonstrates the implementation of three MNIST classifiers:
1. Random Forest
2. Feed-Forward Neural Network
3. Convolutional Neural Network (CNN)

Each model implements the `MnistClassifierInterface` with methods:
- `train`
- `predict`

In [1]:
# ================================
# 1. Import libraries
# ================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy
import random
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torchvision import datasets, transforms
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
from abc import ABC, abstractmethod

In [2]:
# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
# ================================
# 2. Data preparation
# ================================

# Transform images to tensors for PyTorch models
transform_train = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="data", train=True, download=True, transform=transform_train)
test_dataset  = datasets.MNIST(root="data", train=False, download=True, transform=transform_test)

# DataLoader for NN and CNN
train_subset, val_subset = random_split(train_dataset, [54000, 6000],
                               generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_subset,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

# For RandomForest — convert to numpy arrays
X_train_rf = train_dataset.data.view(-1, 28*28).numpy().astype(np.float32) / 255.0
y_train_rf = train_dataset.targets.numpy().astype(np.int64)
X_test_rf  = test_dataset.data.view(-1, 28*28).numpy().astype(np.float32) / 255.0
y_test_rf  = test_dataset.targets.numpy().astype(np.int64)

100%|██████████| 9.91M/9.91M [00:00<00:00, 51.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.72MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.6MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]


In [4]:
# ================================
# 3. MnistClassifierInterface
# ================================
class MnistClassifierInterface(ABC):
    @abstractmethod
    def fit(self, train_data, train_labels=None):
        pass

    @abstractmethod
    def predict(self, x):
        pass

In [ ]:
# ================================
# 4. Random Forest Model with Randomized Search
# ================================

# Random search parameters
param_dist = {
    'n_estimators': [150, 200],
    'max_depth': [None, 30],
    'max_features': ['sqrt'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True],
    'criterion': ['gini', 'entropy']
}

# Model
rf = RandomForestClassifier(random_state=42)

# Using RandomizedSearchCV is faster because it only tests n_iter combinations
random_search = RandomizedSearchCV(estimator=rf,
                                   param_distributions=param_dist,
                                   n_iter=20,  # 20 random combinations
                                   cv=3,
                                   scoring='accuracy',
                                   verbose=1,
                                   n_jobs=-1,
                                   random_state=42)

# Model training
random_search.fit(X_train_rf, y_train_rf)

# Result
print("The best parameters:", random_search.best_params_)
print("Best accuracy (CV):", random_search.best_score_)

# Test set score
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test_rf)
test_acc = accuracy_score(y_test_rf, y_pred)
print(f"Accuracy on the test: {test_acc:.4f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
The best parameters: {'n_estimators': 150, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None, 'criterion': 'gini', 'bootstrap': True}
Best accuracy (CV): 0.9655666666666667
Accuracy on the test: 0.9706


In [ ]:
# ================================
# Random Forest Implementation
# ================================

class RandomForestMnist(MnistClassifierInterface):
    def __init__(self, **rf_params):
        self.model = RandomForestClassifier(
            n_estimators=150, min_samples_split=2, min_samples_leaf=1,
            max_features='sqrt', max_depth= None,
            criterion='gini', bootstrap=True,
            random_state=42
        )

    def fit(self, train_data, train_labels=None, **kwargs):
        if train_labels is None:
            raise ValueError("train_labels required for RandomForestMnist.fit")
        self.model.fit(train_data, train_labels)

    def predict(self, x, **kwargs):
        return self.model.predict(x)

In [5]:
# ================================
# 5. Base class for trainable PyTorch models with training, validation, and early stopping
# ================================

class TrainableNN(nn.Module, MnistClassifierInterface):
    def __init__(self):
        super().__init__()

    def fit(self, train_data, val_data=None, epochs=50, lr=0.001, device="cpu", early_stopping_patience=5):
        self.to(device)
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()

        best_loss = float('inf')
        best_model_wts = copy.deepcopy(self.state_dict())
        patience_counter = 0

        for epoch in range(epochs):
            # Train
            self.train()
            running_loss = 0.0
            num_batches = 0
            for images, labels in train_data:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = self(images)
                loss = loss_fn(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                num_batches += 1
            train_loss = running_loss / max(1, num_batches)

            # Validation
            if val_data is not None:
                self.eval()
                val_loss = 0.0
                val_batches = 0
                with torch.no_grad():
                    for images, labels in val_data:
                        images, labels = images.to(device), labels.to(device)
                        outputs = self(images)
                        val_loss += loss_fn(outputs, labels).item()
                        val_batches += 1
                val_loss /= max(1, val_batches)

                print(f"Epoch [{epoch+1}/{epochs}] — Train Loss: {train_loss:.4f} — Val Loss: {val_loss:.4f}")

                # Save best
                if val_loss < best_loss:
                    best_loss = val_loss
                    best_model_wts = copy.deepcopy(self.state_dict())
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= early_stopping_patience:
                        print(f"Early stopping at epoch {epoch+1}")
                        break
            else:
                print(f"Epoch [{epoch+1}/{epochs}] — Train Loss: {train_loss:.4f}")

        # Restore best
        self.load_state_dict(best_model_wts)
        print(f"\nRestored best model (Val Loss: {best_loss:.4f})")
        checkpoint_path = f"{self.__class__.__name__}_best.pth"   # FeedForwardNN_best.pth або CNNClassifierKerasStyle_best.pth
        torch.save(best_model_wts, checkpoint_path)
        print(f"Saved to {checkpoint_path}")

    def predict(self, x, device="cpu"):
        # Make predictions on a batch or full dataset
        self.eval()
        self.to(device)
        x = x.to(device)
        with torch.no_grad():
            outputs = self(x)
            return torch.argmax(outputs, dim=1)

In [ ]:
# ================================
# 6. Feed-Forward Neural Network
# ================================

class FeedForwardNN(TrainableNN):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 10)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        return self.fc3(x)

In [6]:
# ================================
# 7. Convolutional Neural Network
# ================================

class CNNClassifierKerasStyle(TrainableNN):
    def __init__(self):
        super().__init__()
        # --- Convolutional layers ---
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # --- Pooling ---
        self.pool = nn.MaxPool2d(2, 2)

        # --- Dropout ---
        self.dropout_conv = nn.Dropout(0.25)
        self.dropout_fc = nn.Dropout(0.5)

        # --- Fully connected ---
        self.fc1 = nn.Linear(128 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.dropout_conv(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)
        return self.fc2(x)

In [7]:
# ==================
# 8. MnistClassifier
# ==================

class MnistClassifier:
    def __init__(self, algorithm):
        if algorithm == "rf":
            self.model = RandomForestMnist()
        elif algorithm == "nn":
            self.model = FeedForwardNN()
        elif algorithm == "cnn":
            self.model = CNNClassifierKerasStyle()
        else:
            raise ValueError("Unknown algorithm")

    def fit(self, train_data, train_labels=None, **kwargs):
      if isinstance(self.model, TrainableNN):
        self.model.fit(train_data, **kwargs)
      else:
        self.model.fit(train_data, train_labels)

    def predict(self, x, **kwargs):
        return self.model.predict(x, **kwargs)

In [8]:
# ================================
# 9. Evaluation Function
# ================================

def evaluate(classifier: MnistClassifier, loader, device="cpu"):
    correct, total = 0, 0
    for images, labels in loader:
        images = images.to(device)
        preds = classifier.predict(images, device=device)
        correct += (preds == labels.to(device)).sum().item()
        total += labels.size(0)
    return correct / total

In [ ]:
# ================================
# 10. Example Usage
# ================================

# ---- Random Forest ----
rf = RandomForestMnist()
rf.fit(X_train_rf, y_train_rf)

rf_preds = rf.predict(X_test_rf)
rf_acc = accuracy_score(y_test_rf, rf_preds)
print(f"Random Forest Accuracy: {rf_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test_rf, rf_preds))

Random Forest Accuracy: 0.9706

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       980
           1       0.99      0.99      0.99      1135
           2       0.96      0.97      0.97      1032
           3       0.96      0.96      0.96      1010
           4       0.98      0.97      0.98       982
           5       0.97      0.97      0.97       892
           6       0.98      0.98      0.98       958
           7       0.97      0.96      0.97      1028
           8       0.96      0.96      0.96       974
           9       0.96      0.95      0.96      1009

    accuracy                           0.97     10000
   macro avg       0.97      0.97      0.97     10000
weighted avg       0.97      0.97      0.97     10000



Random Forest serves as a baseline classical ML model.
The model performs very well with an overall accuracy of 97% on the test set.

The F1-scores for all classes range between 0.96 and 0.99, indicating balanced precision and recall across digits.

Some digits are easier to recognize, e.g., 0, 1, and 6 have high recall values (>0.98), meaning the model correctly identifies almost all instances of these digits.

Some digits are slightly more challenging, e.g., 9 has a recall of 0.95, showing that it is occasionally misclassified.

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# --- Feed-Forward Neural Network ---

nn_model = MnistClassifier("nn")
nn_model.fit(train_loader, val_data=val_loader,
             epochs=30, lr=0.001, device=device, early_stopping_patience=7)
nn_acc = evaluate(nn_model, test_loader, device=device)
print(f"Feed-Forward NN Accuracy: {nn_acc:.4f}")

Epoch [1/30] — Train Loss: 0.5198 — Val Loss: 0.2240
Epoch [2/30] — Train Loss: 0.2950 — Val Loss: 0.1713
Epoch [3/30] — Train Loss: 0.2446 — Val Loss: 0.1490
Epoch [4/30] — Train Loss: 0.2218 — Val Loss: 0.1241
Epoch [5/30] — Train Loss: 0.2016 — Val Loss: 0.1263
Epoch [6/30] — Train Loss: 0.1905 — Val Loss: 0.1176
Epoch [7/30] — Train Loss: 0.1790 — Val Loss: 0.1039
Epoch [8/30] — Train Loss: 0.1739 — Val Loss: 0.1045
Epoch [9/30] — Train Loss: 0.1624 — Val Loss: 0.0925
Epoch [10/30] — Train Loss: 0.1599 — Val Loss: 0.0940
Epoch [11/30] — Train Loss: 0.1557 — Val Loss: 0.0898
Epoch [12/30] — Train Loss: 0.1479 — Val Loss: 0.0883
Epoch [13/30] — Train Loss: 0.1420 — Val Loss: 0.0797
Epoch [14/30] — Train Loss: 0.1400 — Val Loss: 0.0803
Epoch [15/30] — Train Loss: 0.1387 — Val Loss: 0.0830
Epoch [16/30] — Train Loss: 0.1366 — Val Loss: 0.0781
Epoch [17/30] — Train Loss: 0.1313 — Val Loss: 0.0746
Epoch [18/30] — Train Loss: 0.1295 — Val Loss: 0.0789
Epoch [19/30] — Train Loss: 0.1267 — 

Training loss decreased from 0.5198 to 0.1089, indicating good learning of the patterns in the training set.

Validation loss reached its minimum at 0.0589, showing the model generalizes well to unseen data without significant overfitting.

The final accuracy on the test set is 99.02%.

In [11]:
# --- Convolutional Neural Network ---

cnn_model = MnistClassifier("cnn")
cnn_model.fit(train_loader, val_data=val_loader,
              epochs=100, lr=0.0008, device=device, early_stopping_patience=10)
cnn_acc = evaluate(cnn_model, test_loader, device=device)
print(f"CNN Accuracy: {cnn_acc:.4f}")

Epoch [1/100] — Train Loss: 0.2730 — Val Loss: 0.1213
Epoch [2/100] — Train Loss: 0.1214 — Val Loss: 0.0941
Epoch [3/100] — Train Loss: 0.0996 — Val Loss: 0.0533
Epoch [4/100] — Train Loss: 0.0900 — Val Loss: 0.0505
Epoch [5/100] — Train Loss: 0.0772 — Val Loss: 0.0501
Epoch [6/100] — Train Loss: 0.0722 — Val Loss: 0.0445
Epoch [7/100] — Train Loss: 0.0676 — Val Loss: 0.0401
Epoch [8/100] — Train Loss: 0.0630 — Val Loss: 0.0451
Epoch [9/100] — Train Loss: 0.0542 — Val Loss: 0.0417
Epoch [10/100] — Train Loss: 0.0546 — Val Loss: 0.0404
Epoch [11/100] — Train Loss: 0.0522 — Val Loss: 0.0422
Epoch [12/100] — Train Loss: 0.0489 — Val Loss: 0.0391
Epoch [13/100] — Train Loss: 0.0458 — Val Loss: 0.0308
Epoch [14/100] — Train Loss: 0.0465 — Val Loss: 0.0345
Epoch [15/100] — Train Loss: 0.0401 — Val Loss: 0.0283
Epoch [16/100] — Train Loss: 0.0420 — Val Loss: 0.0317
Epoch [17/100] — Train Loss: 0.0402 — Val Loss: 0.0316
Epoch [18/100] — Train Loss: 0.0375 — Val Loss: 0.0292
Epoch [19/100] — Tr

## Results Summary

| Model | Test Accuracy |
|--------|----------------|
| Random Forest | 0.9706 |
| Feed-Forward NN | 0.9902 |
| CNN | 0.995 |

CNN achieved the highest accuracy on MNIST, confirming the advantage of convolutional architectures for image data.

## Edge Cases

1. **Empty input tensor** → model should raise a clear error instead of crashing.  
2. **Wrong image size (e.g., 32×32)** → should be reshaped or rejected before training.  
3. **Too few epochs** → underfitting, accuracy <90%.  
4. **Too high learning rate** → unstable training, loss oscillates.  
5. **No early stopping** → risk of overfitting after ~20 epochs.

In [12]:
# Edge case:

# 1. Empty input tensor — expect runtime error
try:
    cnn_model.predict(torch.tensor([]))
except Exception as e:
    print("Handled empty input:", e)

Handled empty input: Expected 3D (unbatched) or 4D (batched) input to conv2d, but got input of size: [0]


In [13]:
# 2. Wrong image size (32x32) — should trigger shape mismatch
wrong_input = torch.randn(1, 1, 32, 32)
try:
    cnn_model.predict(wrong_input)
except Exception as e:
    print("Handled wrong shape:", e)

Handled wrong shape: mat1 and mat2 shapes cannot be multiplied (1x8192 and 6272x256)


In [14]:
# 3. Too high learning rate
small_cnn = MnistClassifier("cnn")
small_cnn.fit(train_loader, val_data=test_loader, epochs=3, lr=0.1, device=device)

Epoch [1/3] — Train Loss: 3.7911 — Val Loss: 2.3156
Epoch [2/3] — Train Loss: 2.3107 — Val Loss: 2.3074
Epoch [3/3] — Train Loss: 2.3100 — Val Loss: 2.3054

Restored best model (Val Loss: 2.3054)
Saved to CNNClassifierKerasStyle_best.pth
